# EDA Report — TCGA Gene Expression Dataset

This notebook performs the exploratory data analysis for the **TCGA gene-expression dataset** used in the project **“Visualizing Genomic Features for Disease Subtype Cluster Analysis.”**

### Analysis covered
- Dataset structure and variable classification
- Missing values and duplicates
- Descriptive statistics
- Gene-expression distributions
- Feature variability
- Gene–gene correlation
- Data preparation for PCA and t-SNE
- PCA visualization
- t-SNE visualization
- K-Means clustering
- Hierarchical clustering

> **Important:** The uploaded CSV is arranged as **genes × samples**. Therefore, the notebook transposes the expression matrix before ML analysis so that samples are rows and genes are features.


In [ ]:
# Import libraries
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

print("Libraries imported successfully.")


In [ ]:
# Load dataset
# Put RNA_seq_1.csv in the same folder as this notebook before running.
DATA_FILE = "RNA_seq_1.csv"

if not os.path.exists(DATA_FILE):
    # If running in the uploaded-file environment, use the known uploaded path.
    DATA_FILE = r'/mnt/data/tcga_extract/RNA_seq_1.csv'

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 1. Dataset Overview

The original file contains **genes as rows** and **TCGA samples as columns**.

- `attrib_name` → gene identifier
- Remaining columns → TCGA sample expression measurements
- Numerical values → gene-expression measurements


In [ ]:
# Basic structure
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names (first 10):")
print(df.columns[:10].tolist())

print("\nData types:")
display(df.dtypes.value_counts())

print("\nUnique gene identifiers:", df["attrib_name"].nunique())


In [ ]:
# Variable classification
gene_column = "attrib_name"
sample_columns = [c for c in df.columns if c != gene_column]

print("Categorical/text variable:")
print(f"  {gene_column} — identifies the gene")

print("\nNumerical variables:")
print(f"  {len(sample_columns)} TCGA sample columns")

print("\nML interpretation:")
print("  Features: gene-expression measurements")
print("  Target: no predefined target for unsupervised clustering")


## 2. Data Quality — Missing Values and Duplicates

In [ ]:
# Missing values
missing_total = int(df.isna().sum().sum())
missing_by_column = df.isna().sum().sort_values(ascending=False)

print("Total missing values:", missing_total)
print("\nColumns with missing values:", int((missing_by_column > 0).sum()))

if missing_total > 0:
    display(missing_by_column[missing_by_column > 0].head(20).to_frame("missing_values"))
else:
    print("No missing values found.")


In [ ]:
# Duplicate analysis
duplicate_rows = int(df.duplicated().sum())
duplicate_genes = int(df[gene_column].duplicated().sum())

print("Duplicate rows:", duplicate_rows)
print("Duplicate gene identifiers:", duplicate_genes)


## 3. Descriptive Statistics

Descriptive statistics help us understand the scale and variability of gene-expression values.


In [ ]:
# Numerical descriptive statistics
expression = df[sample_columns].apply(pd.to_numeric, errors="coerce")

stats = expression.describe().T
display(stats.head(10))

print("Overall expression summary:")
print("Mean:", expression.stack().mean())
print("Std:", expression.stack().std())
print("Min:", expression.stack().min())
print("Median:", expression.stack().median())
print("Max:", expression.stack().max())


## 4. Gene-Expression Distributions

To keep the EDA readable, the plots below use a small number of representative genes rather than thousands of separate plots.


In [ ]:
# Select representative genes from the dataset
representative_n = min(6, len(df))
representative_genes = df[gene_column].head(representative_n).tolist()

# Plot each representative gene separately
for gene in representative_genes:
    row = df.loc[df[gene_column] == gene, sample_columns].apply(pd.to_numeric, errors="coerce").iloc[0].dropna()
    plt.figure(figsize=(8, 4))
    plt.hist(row, bins=25)
    plt.title(f"Expression Distribution — {gene}")
    plt.xlabel("Expression value")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 5. Feature Variability

Variance helps identify genes whose expression changes substantially across samples.


In [ ]:
# Variance of each gene across samples
gene_variance = expression.var(axis=1)
variance_table = pd.DataFrame({
    "gene": df[gene_column],
    "variance": gene_variance
}).sort_values("variance", ascending=False)

print("Top 10 most variable genes:")
display(variance_table.head(10))


In [ ]:
# Box plot of the top 10 most variable genes
top_genes = variance_table.head(10)["gene"].tolist()
plot_data = []

for gene in top_genes:
    vals = df.loc[df[gene_column] == gene, sample_columns].apply(pd.to_numeric, errors="coerce").iloc[0]
    for value in vals.dropna():
        plot_data.append((gene, value))

plot_df = pd.DataFrame(plot_data, columns=["Gene", "Expression"])

plt.figure(figsize=(12, 5))
sns.boxplot(data=plot_df, x="Gene", y="Expression")
plt.xticks(rotation=45, ha="right")
plt.title("Expression of Top 10 Most Variable Genes")
plt.tight_layout()
plt.show()


## 6. Gene–Gene Correlation

Correlation is calculated between genes across the TCGA samples.

Because the full dataset contains thousands of genes, a manageable subset is used for the heatmap.


In [ ]:
# Select 20 most variable genes for correlation heatmap
corr_genes = variance_table.head(min(20, len(variance_table)))["gene"].tolist()

# Rows = genes, columns = samples; transpose to samples × genes
gene_matrix_small = (
    df.set_index(gene_column)
      .loc[corr_genes]
      .apply(pd.to_numeric, errors="coerce")
)

corr_matrix = gene_matrix_small.T.corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Gene–Gene Correlation Heatmap")
plt.tight_layout()
plt.show()


## 7. Prepare Data for PCA, t-SNE and Clustering

For machine learning, the expression matrix is transposed:

**Original:** genes × samples  
**ML format:** samples × genes

Missing values are imputed using the feature mean, and features are standardized before PCA/clustering.


In [ ]:
# Create samples × genes matrix
X = df.set_index(gene_column).apply(pd.to_numeric, errors="coerce").T

print("ML matrix shape (samples × genes):", X.shape)

# Impute missing values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("After preprocessing:")
print("  Missing values remaining:", np.isnan(X_scaled).sum())
print("  Matrix shape:", X_scaled.shape)


## 8. PCA Analysis

PCA reduces the high-dimensional genomic feature space into a smaller number of components while preserving major patterns in the data.


In [ ]:
# PCA: first 2 components
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

print("Explained variance ratio:")
print("PC1:", round(pca_2d.explained_variance_ratio_[0], 4))
print("PC2:", round(pca_2d.explained_variance_ratio_[1], 4))
print("Total (PC1 + PC2):", round(pca_2d.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA — TCGA Gene Expression Samples")
plt.tight_layout()
plt.show()


## 9. t-SNE Visualization

t-SNE creates a low-dimensional representation that emphasizes local relationships between samples.


In [ ]:
# Use PCA as a preprocessing step for t-SNE to reduce computational cost
n_pca = min(50, X_scaled.shape[0] - 1, X_scaled.shape[1])
pca_50 = PCA(n_components=n_pca, random_state=42)
X_pca50 = pca_50.fit_transform(X_scaled)

# Perplexity must be smaller than number of samples
perplexity = min(30, max(5, (X_pca50.shape[0] - 1) // 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    random_state=42,
    init="pca",
    learning_rate="auto"
)
X_tsne = tsne.fit_transform(X_pca50)

plt.figure(figsize=(8, 6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.7)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE — TCGA Gene Expression Samples")
plt.tight_layout()
plt.show()


## 10. K-Means Clustering

K-Means groups samples into clusters based on similarity.

The code evaluates a range of cluster counts using the silhouette score and then selects the best-scoring value.


In [ ]:
# Evaluate K-Means cluster counts
max_k = min(8, X_pca50.shape[0] - 1)
k_values = range(2, max_k + 1)

silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca50)
    silhouette_scores.append(silhouette_score(X_pca50, labels))

k_results = pd.DataFrame({
    "k": list(k_values),
    "silhouette_score": silhouette_scores
})

display(k_results)

best_k = int(k_results.loc[k_results["silhouette_score"].idxmax(), "k"])
print("Selected K:", best_k)


In [ ]:
# Fit final K-Means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca50)

# Visualize K-Means clusters on t-SNE coordinates
plt.figure(figsize=(8, 6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap="tab10", alpha=0.8)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title(f"K-Means Clusters (k={best_k}) on t-SNE Map")
plt.tight_layout()
plt.show()

print("Cluster counts:")
print(pd.Series(kmeans_labels).value_counts().sort_index())


## 11. Hierarchical Clustering

Hierarchical (agglomerative) clustering builds groups by progressively joining similar samples.

For comparison with K-Means, the same selected number of clusters is used.


In [ ]:
# Agglomerative clustering
hier = AgglomerativeClustering(n_clusters=best_k)
hier_labels = hier.fit_predict(X_pca50)

hier_score = silhouette_score(X_pca50, hier_labels)

print("Hierarchical clustering silhouette score:", round(hier_score, 4))
print("\nCluster counts:")
print(pd.Series(hier_labels).value_counts().sort_index())


In [ ]:
# Visualize hierarchical clusters on t-SNE coordinates
plt.figure(figsize=(8, 6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=hier_labels, cmap="tab10", alpha=0.8)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title(f"Hierarchical Clusters (k={best_k}) on t-SNE Map")
plt.tight_layout()
plt.show()


## 12. Final EDA Observations

- The dataset is **high-dimensional**, containing thousands of genomic features.
- `attrib_name` identifies the genes, while the TCGA sample columns contain numerical expression measurements.
- Missing values are checked and handled before machine-learning analysis.
- Gene-expression distributions and variance show that genes differ in their level of variability.
- Gene–gene correlation helps identify related expression patterns.
- PCA provides a compact representation of the major patterns.
- t-SNE provides a 2D visualization of local sample relationships.
- K-Means and Hierarchical Clustering group samples based on genomic similarity.
- There is **no predefined target variable** for the unsupervised clustering task.

### Project pipeline

**TCGA Gene Expression → EDA → Preprocessing → PCA → t-SNE → K-Means / Hierarchical Clustering → Candidate Sample Groups**
